# SmartHeart Colab Training

## 1. Clone the repository

Clone the latest `main` branch into the Colab workspace. If it is already present, update it with a fast-forward pull, then make it the active working directory.

In [1]:
!if [ -d /content/smart-heart/.git ]; then git -C /content/smart-heart pull --ff-only origin main; else git clone --branch main --single-branch https://github.com/Bojan-Ivanovski/smart-heart.git /content/smart-heart; fi
%cd /content/smart-heart

Cloning into '/content/smart-heart'...
remote: Enumerating objects: 2521, done.
remote: Counting objects: 100% (1724/1724), done.
remote: Compressing objects: 100% (431/431), done.
remote: Total 2521 (delta 1355), reused 1649 (delta 1291), pack-reused 797 (from 2)
Receiving objects: 100% (2521/2521), 120.38 MiB | 19.92 MiB/s, done.
Resolving deltas: 100% (1632/1632), done.
Filtering content: 100% (5/5), 292.74 MiB | 5.09 MiB/s, done.
/content/smart-heart


### Optional: Download trained checkpoints

Run this cell only when you want to validate or generate predictions with the committed checkpoints without training them again. It installs Git LFS when necessary and replaces the small pointer files with the complete model weights.

In [2]:
!if ! command -v git-lfs >/dev/null 2>&1; then apt-get update -qq && apt-get install -y git-lfs; fi
!git lfs install --local
!git lfs pull

Updated Git hooks.
Git LFS initialized.


## 2. Configure the runtime

Select an **A100 GPU** from **Runtime > Change runtime type**. Choose the Hugging Face base model and execution device here; the curriculum stages define their own window sizes and always use the complete canonical dataset.

In [3]:
import os

DEVICE = "cuda"  # @param ["cuda", "xla", "cpu"]
MODEL_ID = "google/gemma-3-1b-pt"  # @param {type:"string"}

os.environ["SMARTHEART_DEVICE"] = DEVICE
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
if DEVICE == "xla":
    os.environ["PJRT_DEVICE"] = "TPU"

print(f"Model: {MODEL_ID}")
print(f"Device: {DEVICE}")
print("Architecture: OpenTSLM SP with LoRA")
print("Dataset: complete canonical dataset")

Model: google/gemma-3-1b-pt
Device: cuda
Architecture: OpenTSLM SP with LoRA
Dataset: complete canonical dataset


## 3. Install dependencies

Upgrade `pip`, remove Colab packages that conflict with OpenTSLM's pinned Hugging Face stack, and install the repository environment. TPU runs use the additional XLA requirements.

In [4]:
%%bash
set -euo pipefail
python -m pip install --upgrade pip
python -m pip uninstall --yes diffusers gradio
if [[ "${SMARTHEART_DEVICE:-cuda}" == "xla" ]]; then
    python -m pip install -r requirements-tpu.txt
else
    python -m pip install -r requirements.txt
fi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 83.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Found existing installation: diffusers 0.40.0
Uninstalling diffusers-0.40.0:
  Successfully uninstalled diffusers-0.40.0
Found existing installation: gradio 6.26.0
Uninstalling gradio-6.26.0:
  Successfully uninstalled gradio-6.26.0
  Cloning https://github.com/StanfordBDHG/OpenTSLM.git (to revision refs/pull/47/head) to /tmp/pip-install-_5l_5vk5/opentslm_a4eb30900edb48ee9caf05172232e0f7
  Did not find branch or tag 'refs/pull/47/head', assuming revision or ref.
  Resolved https://github.com/StanfordBDHG/OpenTSLM.git to commit 3356bbd4ecd2a88f713784d3e14f71a2b5b5b2f2
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with stat

  Running command git clone --filter=blob:none --quiet https://github.com/StanfordBDHG/OpenTSLM.git /tmp/pip-install-_5l_5vk5/opentslm_a4eb30900edb48ee9caf05172232e0f7
  Running command git fetch -q https://github.com/StanfordBDHG/OpenTSLM.git refs/pull/47/head
  Running command git checkout -q 3356bbd4ecd2a88f713784d3e14f71a2b5b5b2f2


## 4. Authenticate with Hugging Face

Gemma access requires accepting its license on Hugging Face. Add a Colab secret named `HF_TOKEN` using the key icon in the left sidebar and grant this notebook access. Authenticate before running any training, validation, or prediction command.

In [5]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add an HF_TOKEN secret and grant this notebook access.")

login(token=hf_token, add_to_git_credential=False)
print("Hugging Face authentication configured.")

Hugging Face authentication configured.


## 5. Validate the dataset

Parse every patient through the Pydantic contracts and summarize cohort balance, diagnoses, runtime splits, usable recordings and segments, curriculum supervision, signal storage, and default-policy window counts.

In [6]:

  %%bash
  set -euo pipefail
  test -d dataset/patients
  echo "Patients: $(find dataset/patients -mindepth 1 -maxdepth 1 -type d | wc -l)"
  echo "Signal archives: $(find dataset/patients -type f -name 'signals.npz' | wc -l)"


Patients: 210
Signal archives: 210


## 6. Validate the environment and preview data

Report the installed PyTorch version and CUDA availability, then preview a Stage 1 sample using its segment's default window policy to confirm discovery, signal loading, feature analysis, and prompt construction.

In [7]:
!python -c "import torch; print('torch:', torch.__version__); print('cuda_available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"
!python -m src.main preview --stage stage1_mcq

torch: 2.9.0+cu128
cuda_available: True
device: NVIDIA A100-SXM4-80GB
stage: stage1_mcq
stage_samples: 30190
signal_windows: 15095
patient: children__v107
recordings: visual-attention-01
segments: visual-attention-01__segment-001
split: train
source: adhd_children

PRE-PROMPT
Analyze the EEG time series and answer the multiple-choice question using the measured signal pattern.

TIME SERIES
[1] shape=(512,) Recording visual-attention-01, segment visual-attention-01__segment-001, channel Fz, contains raw_eeg from the visual_attention_cartoon_counting condition. Its original mean is 118.1211 and standard deviation is 850.2523.
[2] shape=(512,) Recording visual-attention-01, segment visual-attention-01__segment-001, channel Cz, contains raw_eeg from the visual_attention_cartoon_counting condition. Its original mean is 110.6367 and standard deviation is 1160.3333.
[3] shape=(512,) Recording visual-attention-01, segment visual-attention-01__segment-001, channel Pz, contains raw_eeg from the 

## 7. Train the model

Run the five stages in order. Every command uses the complete training split and saves its own checkpoint. Stage 1 starts a clean curriculum; every later stage automatically initializes from the preceding checkpoint.

### Stage 1: MCQ warmup

Train short, objective frequency-analysis answers using large single-window inputs.

**Question**

Which listed frequency band has the greatest relative power?

> A. Delta &nbsp;&nbsp; B. Theta &nbsp;&nbsp; C. Alpha &nbsp;&nbsp; D. Beta

**Input**

> EEG window: Cz channel, 4,096 samples. Relative power: delta 42%, theta 25%, alpha 18%, beta 10%, gamma 5%.

**Expected answer:** `A`

In [14]:
!python -u -m src.main train --model-id "{MODEL_ID}" --architecture sp --device "{DEVICE}" --stage stage1_mcq --window-size 1024 --batch-size 1  --epochs 2 --learning-rate 5e-5 --fresh-start

[train] loading model=google/gemma-3-1b-pt runtime=cuda
2026-09-08 14:35:28.080206: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-08 14:35:28.151164: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
✅ LoRA enabled

### Checkpoint after Stage 1

Confirm that the MCQ checkpoint was written before continuing.

In [15]:
!find checkpoints -type f -name '*.pt' -printf '%p (%k KB)\n' | sort

checkpoints/google__gemma-3-1b-pt__fa56a704/sp/stage1_mcq.pt (59952 KB)


### Stage 2: EEG captioning

Continue from Stage 1 and train concise, window-level descriptions of spectral, temporal, and quality features.

**Question**

> Describe the EEG time series objectively in one concise clinical-style caption. Do not infer a diagnosis from this segment.

**Input**

> EEG window: F4 channel, 4,096 samples. Relative power: delta 21%, theta 38%, alpha 19%, beta 14%, gamma 8%; temporal variability: moderate; zero-crossing rate: 0.083; detected artifact fraction: 0.6%.

**Expected answer**

> This window is dominated by theta-band activity. Relative band power is delta 21.0%, theta 38.0%, alpha 19.0%, beta 14.0%, gamma 8.0%. Temporal behavior is moderate, with a zero-crossing rate of 0.083. Signal screening found minimal detected high-amplitude artifact (0.6%).

In [17]:
!python -u -m src.main train --model-id "{MODEL_ID}" --architecture sp --device "{DEVICE}" --stage stage2_eeg_captioning --window-size 1024 --batch-size 1 --epochs 3 --learning-rate 5e-5

[train] loading model=google/gemma-3-1b-pt runtime=cuda
2026-09-08 14:49:26.265358: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-08 14:49:26.335057: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
✅ LoRA enabled

### Checkpoint after Stage 2

Confirm that both completed curriculum snapshots are available.

In [18]:
!find checkpoints -type f -name '*.pt' -printf '%p (%k KB)\n' | sort

checkpoints/google__gemma-3-1b-pt__fa56a704/sp/stage1_mcq.pt (59952 KB)
checkpoints/google__gemma-3-1b-pt__fa56a704/sp/stage2_eeg_captioning.pt (59956 KB)


### Stage 3: Attention-task reasoning

Continue from Stage 2 and combine representative segments into recording-level attention-task reasoning.

**Question**

> Analyze this attention-task EEG recording. Provide a concise reasoning chain followed by a conclusion and confidence level.

**Input**

> Representative windows from an attention-task recording show elevated theta relative to beta, moderate temporal variability, and low artifact burden across Cz and F4.

**Expected answer**

> Observations: Theta activity is consistently prominent; signal quality is usable. Reasoning: The repeated pattern supports reduced task-state cortical activation, but EEG alone is not diagnostic. Conclusion: The recording shows an attention-task pattern consistent with reduced sustained-attention engagement. Confidence: moderate.

In [24]:
!python -u -m src.main train --model-id "{MODEL_ID}" --architecture sp --device "{DEVICE}" --stage stage3_attention_task_cot --window-size 1024 --batch-size 1 --epochs 8 --learning-rate 2e-5

[train] loading model=google/gemma-3-1b-pt runtime=cuda
2026-09-08 14:53:25.758003: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-08 14:53:25.828697: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
✅ LoRA enabled

### Checkpoint after Stage 3

Confirm that the attention-task reasoning snapshot was written.

In [25]:
!find checkpoints -type f -name '*.pt' -printf '%p (%k KB)\n' | sort

checkpoints/google__gemma-3-1b-pt__fa56a704/sp/stage1_mcq.pt (59952 KB)
checkpoints/google__gemma-3-1b-pt__fa56a704/sp/stage2_eeg_captioning.pt (59956 KB)
checkpoints/google__gemma-3-1b-pt__fa56a704/sp/stage3_attention_task_cot.pt (59956 KB)


### Stage 4: Resting-state reasoning

Continue from Stage 3 and train recording-level reasoning about resting-state EEG patterns.

**Question**

> Analyze this resting-state EEG recording. Provide a concise reasoning chain followed by a conclusion and confidence level without using EEG alone to make a diagnosis.

**Input**

> Representative windows from one eyes-closed resting-state recording show stable background activity, posterior alpha dominance, and minimal artifact.

**Expected answer**

> Observations: Alpha activity is dominant during eyes-closed rest; temporal behavior remains stable; artifact burden is low. Reasoning: The spectral and temporal pattern is physiologically plausible and the recording is suitable for interpretation. Conclusion: The resting-state recording shows an organized background pattern. Confidence: high.

In [28]:
!python -u -m src.main train --model-id "{MODEL_ID}" --architecture sp --device "{DEVICE}" --stage stage4_resting_state_cot --window-size 1024 --batch-size 1 --epochs 4 --learning-rate 2e-5

[train] loading model=google/gemma-3-1b-pt runtime=cuda
2026-09-08 14:54:53.144308: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-08 14:54:53.216711: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
✅ LoRA enabled

### Checkpoint after Stage 4

Confirm that the resting-state reasoning snapshot was written.

In [26]:
!find checkpoints -type f -name '*.pt' -printf '%p (%k KB)\n' | sort

checkpoints/google__gemma-3-1b-pt__fa56a704/sp/stage1_mcq.pt (59952 KB)
checkpoints/google__gemma-3-1b-pt__fa56a704/sp/stage2_eeg_captioning.pt (59956 KB)
checkpoints/google__gemma-3-1b-pt__fa56a704/sp/stage3_attention_task_cot.pt (59956 KB)
checkpoints/google__gemma-3-1b-pt__fa56a704/sp/stage4_resting_state_cot.pt (59956 KB)


### Stage 5: Diagnostic reasoning

Continue from Stage 4 and integrate representative patient EEG evidence with the clinical profile. The smaller window controls the combined patient-level sequence length.

**Question**

> Integrate the clinical history and EEG evidence, then give the diagnosis, presentation, severity, and confidence.

**Input**

> EEG evidence: persistent theta prominence during attention tasks with usable resting-state recordings. Clinical profile: 8 inattentive and 3 hyperactive/impulsive symptoms for 18 months, impairment at school and home, concordant parent and teacher reports, and no stronger alternative explanation.

**Expected answer**

> EEG evidence: Task recordings show reproducible theta prominence, used only as supporting evidence. Clinical evidence: Persistent cross-setting inattentive symptoms cause functional impairment and are supported by multiple informants. Alternative explanations: Sleep and anxiety were considered but do not better explain the presentation. Reasoning: The clinical criteria carry the diagnostic conclusion, while EEG modestly supports it. Conclusion: diagnosis ADHD, presentation predominantly inattentive, severity moderate, confidence high.

In [43]:
!python -u -m src.main train --model-id "{MODEL_ID}" --architecture sp --device "{DEVICE}" --stage stage5_diagnostic_cot --window-size 64 --batch-size 1 --epochs 10 --learning-rate 1e-5

[train] loading model=google/gemma-3-1b-pt runtime=cuda
2026-09-08 15:03:32.341299: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-08 15:03:32.414062: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
✅ LoRA enabled

## 8. Inspect final checkpoints

Verify that all five sequential curriculum snapshots exist before evaluation.

In [27]:
!find checkpoints -type f -name '*.pt' -printf '%p (%k KB)\n' | sort

checkpoints/google__gemma-3-1b-pt__fa56a704/sp/stage1_mcq.pt (59952 KB)
checkpoints/google__gemma-3-1b-pt__fa56a704/sp/stage2_eeg_captioning.pt (59956 KB)
checkpoints/google__gemma-3-1b-pt__fa56a704/sp/stage3_attention_task_cot.pt (59956 KB)
checkpoints/google__gemma-3-1b-pt__fa56a704/sp/stage4_resting_state_cot.pt (59956 KB)
checkpoints/google__gemma-3-1b-pt__fa56a704/sp/stage5_diagnostic_cot.pt (59956 KB)


## 9. Validate the checkpoints

Evaluate each checkpoint separately on validation patients only. Window sizes match their corresponding training stages. The test partition remains untouched until the final prediction demonstration.

### Validate Stage 1

Measure MCQ accuracy on the validation partition.

In [44]:
!python -u -m src.main evaluate --model-id "{MODEL_ID}" --architecture sp --device "{DEVICE}" --split validation --stage stage1_mcq --window-size 4096 --batch-size 1

2026-09-08 15:04:06.474489: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-08 15:04:06.545675: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
✅ LoRA enabled:
   LoRA parameters: 13,045,760
   Total trainable para

### Validate Stage 2

Measure caption exact match and token overlap on the validation partition.

In [ ]:
!python -u -m src.main evaluate --model-id "{MODEL_ID}" --architecture sp --device "{DEVICE}" --split validation --stage stage2_eeg_captioning --window-size 4096 --batch-size 1

### Validate Stage 3

Measure attention-task reasoning and conclusion metrics on validation recordings.

In [ ]:
!python -u -m src.main evaluate --model-id "{MODEL_ID}" --architecture sp --device "{DEVICE}" --split validation --stage stage3_attention_task_cot --window-size 2048 --batch-size 1

### Validate Stage 4

Measure resting-state reasoning and conclusion metrics on validation recordings.

In [ ]:
!python -u -m src.main evaluate --model-id "{MODEL_ID}" --architecture sp --device "{DEVICE}" --split validation --stage stage4_resting_state_cot --window-size 2048 --batch-size 1

### Validate Stage 5

Measure diagnostic fields and reasoning quality on validation patients.

In [ ]:
!python -u -m src.main evaluate --model-id "{MODEL_ID}" --architecture sp --device "{DEVICE}" --split validation --stage stage5_diagnostic_cot --window-size 128 --batch-size 1

## 10. Generate one prediction

Load the final diagnostic checkpoint and generate one response for the first patient in the untouched test partition. The prediction command automatically uses Stage 5, its 128-sample windows, and patient-level diagnostic prompting.

In [ ]:
!python -u -m src.main predict --model-id "{MODEL_ID}" --device "{DEVICE}" --index 0